# 00 · Configuración de Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tomas-pucv/Camanchaca-Predict/blob/main/notebooks/00_setup_colab.ipynb)

**Propósito.** Preparar y verificar el entorno: GPU, clon del repositorio en una referencia
fija, instalación del paquete `camanchaca`, variables de entorno y registro del commit usado.

**Tiempo esperado.** ~5 min (+3–6 min si se corre la prueba de humo).
**Entorno.** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4* (para el notebook 01 basta CPU).

**Salidas.** Entorno verificado (`scripts/verify_env.py`) y una línea con fecha, commit, versiones y
GPU en `MyDrive/camanchaca/runs/colab_sessions.txt`.

**Antes de empezar (una vez).** En `MyDrive/camanchaca/` dejar `kaggle.json` y
`cityscapes_credentials.json` (`{"username": "...", "password": "..."}`). **Nunca** subirlos al repo.

Orden de trabajo: `00` → `01` (CPU) → `02` (GPU) → `03` → `04` → `05` → `06`.

## Configuración común
Misma celda en todos los notebooks (detalles en `00_setup_colab`). En **Colab**: monta Drive,
fija `CAMANCHACA_DATA=/content/data` y `CAMANCHACA_RUNS=/content/drive/MyDrive/camanchaca/runs`,
clona/actualiza el repo en la referencia `REF`, instala el paquete (`pip install -e ".[demo,data]"`,
**sin** reinstalar torch) y desempaqueta los `.tar` de datos de Drive (salvo `NEEDS_DATA = False`). **Fuera de Colab** (`SMOKE=True`)
usa datos sintéticos y pesos aleatorios generados por `camanchaca smoke`: sirve para verificar que
todo corre, **sus números no son resultados**.

In [ ]:
NEEDS_DATA = False   # este notebook no usa las imágenes: no desempaquetar los .tar

In [ ]:
# Configuración común (idéntica en los notebooks 00-06)
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SMOKE = not IN_COLAB   # True: datos sintéticos + pesos aleatorios (solo verifica que todo corre)
REF = "main"           # rama/tag/commit del repo; para la entrega final usar "v1.0-final"
REPO_URL = "https://github.com/Tomas-pucv/Camanchaca-Predict.git"
DRIVE_DIR = Path("/content/drive/MyDrive/camanchaca")   # tars de datos, corridas, pesos y credenciales

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["CAMANCHACA_DATA"] = "/content/data"
    os.environ["CAMANCHACA_RUNS"] = str(DRIVE_DIR / "runs")
    REPO = Path("/content/Camanchaca-Predict")
    if not REPO.exists():
        !git clone -q {REPO_URL} {REPO}
    !git -C {REPO} fetch -q --all --tags && git -C {REPO} checkout -q {REF} && (git -C {REPO} pull -q --ff-only 2>/dev/null || true)
    %cd {REPO}
    !pip install -q -e ".[demo,data]"   # NO reinstala torch/torchvision: se usan los de Colab
    if not SMOKE and globals().get("NEEDS_DATA", True):
        for tar in sorted((DRIVE_DIR / "data").glob("*.tar")):   # imágenes preprocesadas (notebook 01)
            !camanchaca unpack --tar {tar}
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    %cd {REPO}
    if shutil.which("camanchaca") is None:
        print("AVISO: falta el comando `camanchaca`: ejecutar  pip install -e '.[dev,demo]'")

WORK = Path(os.environ.get("CAMANCHACA_SMOKE_DIR", str(REPO / "local_workspace" / "nb_smoke")))
if SMOKE:
    os.environ.update(CAMANCHACA_DATA=str(WORK / "data"), CAMANCHACA_RUNS=str(WORK / "runs"),
                      CAMANCHACA_REPORTS=str(WORK / "reports"), HF_HUB_OFFLINE="1")
    if not (WORK / "smoke.yaml").exists():   # ~4-6 min en CPU, una sola vez
        !camanchaca smoke --workdir {WORK}

sys.path.insert(0, str(REPO / "src"))   # importable sin reiniciar el kernel tras `pip install -e`
from camanchaca.config import load_yaml
from camanchaca.paths import data_root, reports_dir, runs_root

EXP = str(WORK / "smoke.yaml") if SMOKE else "configs/experiment/main.yaml"
EXP_NAME = load_yaml(EXP)["name"]
print(f"Colab={IN_COLAB} | SMOKE={SMOKE} | EXP={EXP}")
print(f"datos={data_root()} | corridas={runs_root()} | reportes={reports_dir()}")

## 1. GPU y versiones
Colab trae torch/torchvision con CUDA: **no** reinstalarlos (rompe CUDA). `verify_env.py`
comprueba versiones mínimas (torch ≥ 2.4, timm 1.0.x, gradio 6.x), la GPU y que el paquete
`camanchaca` sea importable.

In [ ]:
!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución -> Cambiar tipo de entorno -> GPU T4"
!{sys.executable} scripts/verify_env.py

## 2. Registrar commit y entorno
Cada sesión deja una línea en Drive: permite citar en el informe exactamente qué código y qué
GPU produjeron cada resultado.

In [ ]:
import datetime, torch, timm
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=REPO).stdout.strip()
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
LINE = (f"{datetime.datetime.now().isoformat(timespec='seconds')} ref={REF} commit={COMMIT} "
        f"torch={torch.__version__} timm={timm.__version__} gpu={GPU}")
SESSIONS = runs_root() / "colab_sessions.txt"
SESSIONS.parent.mkdir(parents=True, exist_ok=True)
with open(SESSIONS, "a", encoding="utf-8") as f: f.write(LINE + "\n")
print(LINE)

## 3. Prueba de humo (opcional, ~3–6 min)
Ejecuta **toda** la cadena (sweep → train-all → predict → baselines → evaluate → latency →
probes → interpret → counterfactual → cars → report) sobre datos sintéticos minúsculos con pesos
aleatorios, en una carpeta temporal. Confirma que el repositorio corre de punta a punta; sus
números **no** son resultados. Fuera de Colab ya corrió en la configuración común.

In [ ]:
RUN_SMOKE = IN_COLAB and not SMOKE
if RUN_SMOKE:
    !camanchaca smoke

## Siguientes pasos
1. `01_prepare_data` (entorno **CPU**): descargas, preprocesamiento, splits y `.tar` en Drive.
2. `02_train` (GPU): barrido de LR y 2 modelos × 3 semillas (re-ejecutable tras desconexiones).
3. `03_evaluate`, `04_interpretability`, `05_cars_vs_fog`: tablas y figuras en `reports/`.
4. `06_demo`: demo Gradio con enlace público y video de respaldo.